In [0]:
config_content = """valid_statuses:
  - Active
  - Inactive
  - Pending

salary_range:
  min: 0
  max: 1000000

thresholds:
  null_warning_pct: 5
  null_fail_pct: 20
"""

volume_path = "/Volumes/workspace/default/day12_files"
with open(f"{volume_path}/config.yaml", "w") as f:
    f.write(config_content)

In [0]:
import yaml

with open(f"{volume_path}/config.yaml") as f:
    config = yaml.safe_load(f)

print(config)
print(config["valid_statuses"])
print(config["thresholds"]["null_warning_pct"])

In [0]:
#Rewrite Day 18's accepted-values check to use config
from pyspark.sql.functions import col, upper

data = [(1, "Active"), (2, "active"), (3, "Inactive"), (4, "Suspended"), (5, "Pending")]
df = spark.createDataFrame(data, ["id", "status"])

valid_statuses_upper = [s.upper() for s in config["valid_statuses"]]
invalid_rows = df.filter(~upper(col("status")).isin(valid_statuses_upper))
invalid_rows.display()

In [0]:
#Rewrite Day 22's threshold flags to use config
from pyspark.sql.functions import when

null_pct_data = [("department", 20.0), ("salary", 3.0), ("name", 0.0)]
report_df = spark.createDataFrame(null_pct_data, ["column", "null_pct"])

report_flagged = report_df.withColumn(
    "status",
    when(col("null_pct") > config["thresholds"]["null_fail_pct"], "FAIL")
    .when(col("null_pct") > config["thresholds"]["null_warning_pct"], "WARNING")
    .otherwise("OK")
)
report_flagged.display()

In [0]:

#Create a second config with different thresholds, and confirm behavior changes
config_strict = """valid_statuses:
  - Active
  - Inactive
  - Pending

salary_range:
  min: 0
  max: 1000000

thresholds:
  null_warning_pct: 1
  null_fail_pct: 5
"""
with open(f"{volume_path}/config_strict.yaml", "w") as f:
    f.write(config_strict)

with open(f"{volume_path}/config_strict.yaml") as f:
    config2 = yaml.safe_load(f)

report_df.withColumn(
    "status",
    when(col("null_pct") > config2["thresholds"]["null_fail_pct"], "FAIL")
    .when(col("null_pct") > config2["thresholds"]["null_warning_pct"], "WARNING")
    .otherwise("OK")
).display()

#Check: with stricter thresholds (1%/5% instead of 5%/20%), department's 20% should now clearly FAIL, and even salary's 3% should now show WARNING instead of OK — same check code, different result, purely from swapping the config file.